In [ ]:
import pickle

prefix = 'data/lba/'
# a list of rdkit mols
all_data = pickle.load(open(f'{prefix}/train_mols.pkl', 'rb'))

print(len(all_data), all_data[0])

In [ ]:
from rdkit import Chem
from tqdm import tqdm
from graph import HyperGraph
import numpy as np

# when ISOMERIC is False, stereochemical information and chirality are not considered.
# turn this to True if you want to build vocab with chirality.
ISOMERIC = False

In [ ]:
hyper_graphs = []

for idx, mol in enumerate(tqdm(all_data)):
    smiles = Chem.MolToSmiles(mol, kekuleSmiles=False, isomericSmiles=ISOMERIC)
    hyper_g = HyperGraph(mol, idx, smiles, isomeric=ISOMERIC)
    hyper_g.contract_basic_tokens()
    hyper_g.check_coverage()
    hyper_graphs.append(hyper_g)

In [ ]:
from vocab_utils import plot_init_node_edge_num

plot_init_node_edge_num(hyper_graphs)

In [ ]:
from vocab_utils import count_atom_freq, count_basic_token_freq, show_vocab_2d, count_hyper_pair_freq

atom_freq = count_atom_freq(hyper_graphs)
print(len(atom_freq))

show_vocab_2d(atom_freq)

bond_freq, aromatic_ring_freq, non_aromatic_ring_freq = count_basic_token_freq(hyper_graphs)
print(len(bond_freq), len(aromatic_ring_freq), len(non_aromatic_ring_freq))

# show_vocab_2d(bond_freq)
# show_vocab_2d(aromatic_ring_freq)
# show_vocab_2d(non_aromatic_ring_freq)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt


af = [v for k, v in atom_freq.most_common()]
bf = [v for k, v in bond_freq.most_common()]
arf = [v for k, v in aromatic_ring_freq.most_common()]
nrf = [v for k, v in non_aromatic_ring_freq.most_common()]
plt.plot(range(len(af)), af, label=f'atom, [{min(af)}, {max(af)}], avg={sum(af) / len(af):.1f}, med={np.median(af):.1f}')
plt.plot(range(len(bf)), bf, label=f'bond, [{min(bf)}, {max(bf)}], avg={sum(bf) / len(bf):.1f}, med={np.median(bf):.1f}')
plt.plot(range(len(arf)), arf, label=f'aromatic ring, [{min(arf)}, {max(arf)}], avg={sum(arf) / len(arf):.1f}, med={np.median(arf):.1f}')
plt.plot(range(len(nrf)), nrf, label=f'non aromatic ring, [{min(nrf)}, {max(nrf)}], avg={sum(nrf) / len(nrf):.1f}, med={np.median(nrf):.1f}')
plt.yscale('log')
plt.xlabel('rank of tokens')
plt.ylabel('log freq')
plt.legend()
plt.show()

In [ ]:
vocab = {}

# takes ~ 2 minutes to greedy search 400 composite tokens
for i in range(400):
    print('-' * 30, f'current vocab size: {len(vocab)}', '-' * 30)
    smiles_counter = count_hyper_pair_freq(hyper_graphs)
    print(smiles_counter.most_common(10))
    candidate = smiles_counter.most_common(1)[0]
    print('contracting', candidate)
    vocab[candidate[0]] = candidate[1]
    affected = []
    for idx, hyper_g in enumerate(tqdm(hyper_graphs)):
        flag = hyper_g.contract_hyper_pair(candidate[0], sharing_hyper_node=True)
        if flag:
            hyper_g.check_coverage()
            affected.append(idx)

    print(len(affected), 'affected!', len(affected) / len(hyper_graphs), '\n')

In [ ]:
print('total # composite tokens:', len(vocab))
from display_utils import show_2d_smiles

last_freq = 1e10
last_item = None
for idx, (k, v) in enumerate(vocab.items()):
    if v > last_freq:
        show_2d_smiles(k)
        print(idx, k, v)
    last_freq = v
    last_item = (k, v)

# show last item in vocab
show_2d_smiles(last_item[0])
print(last_item)


In [ ]:
# display all discovered composite tokens: graph, smiles, and freq
show_vocab_2d(vocab)

In [ ]:
# preserve all atom tokens
atom_vocab = {}
for smiles, freq in atom_freq.most_common():
    atom_vocab[smiles] = freq
show_vocab_2d(atom_vocab)

from collections import Counter

# add bond, aromatic ring, non-aromatic ring, and composite tokens to final vocab
def get_vocab_by_freq(bond_freq: Counter, aromatic_ring_freq: Counter, non_aromatic_ring_freq: Counter, hyper_vocab: dict, min_freq: int = 50):
    print('-' * 30, f'min_freq: {min_freq}', '-' * 30)
    bond_vocab = {}
    aromatic_ring_vocab = {}
    non_aromatic_ring_vocab = {}

    for smiles, freq in bond_freq.most_common():
        if freq > min_freq:
            bond_vocab[smiles] = freq

    for smiles, freq in aromatic_ring_freq.most_common():
        if freq > min_freq:
            aromatic_ring_vocab[smiles] = freq

    for smiles, freq in non_aromatic_ring_freq.most_common():
        if freq > min_freq:
            non_aromatic_ring_vocab[smiles] = freq

    hyper_token_vocab = {}
    for k, v in Counter(hyper_vocab).most_common():
        if v > min_freq:
            hyper_token_vocab[k] = v
    
    vocab_size = len(atom_vocab) + len(bond_vocab) + len(aromatic_ring_vocab) + len(non_aromatic_ring_vocab) + len(hyper_token_vocab)
    print(f'current vocab size: {vocab_size}\n')
    show_vocab_2d(bond_vocab)
    show_vocab_2d(aromatic_ring_vocab)
    show_vocab_2d(non_aromatic_ring_vocab)
    show_vocab_2d(hyper_token_vocab)
    full_vocab = {
        'atom_vocab': atom_vocab,
        'bond_vocab': bond_vocab,
        'aromatic_ring_vocab': aromatic_ring_vocab,
        'non_aromatic_ring_vocab': non_aromatic_ring_vocab,
        'hyper_token_vocab': hyper_token_vocab
    }
    if ISOMERIC:
        pickle.dump(full_vocab, open(f'{prefix}/vocab_iso_{vocab_size}.pkl', 'wb'))
    else:
        pickle.dump(full_vocab, open(f'{prefix}/vocab_{vocab_size}.pkl', 'wb'))

get_vocab_by_freq(bond_freq, aromatic_ring_freq, non_aromatic_ring_freq, vocab, min_freq=50)

## tokenization

In [ ]:
import pickle

prefix = 'data/lba/'
all_data = pickle.load(open(f'{prefix}/train_mols.pkl', 'rb'))

print(len(all_data), all_data[0])

In [ ]:
from tqdm import tqdm
from rdkit import Chem
from graph import HyperGraph
from tokenizer import MergeTokenizer

vocab_path = f'{prefix}/vocab_213.pkl'
tokenizer = MergeTokenizer(vocab_path)

In [ ]:
from display_utils import show_3d_mol

# display tokenization result for the 1st molecule
ISOMERIC = False
idx = 1
mol = all_data[idx]
show_3d_mol(mol)
smiles = Chem.MolToSmiles(mol, kekuleSmiles=False, isomericSmiles=ISOMERIC)
hg = tokenizer.tokenize(mol, idx, smiles)
hg.check_coverage()
hg.show_tokenization()


In [ ]:
from tqdm import tqdm

hyper_graphs = []

for idx, mol in enumerate(tqdm(all_data)):
    smiles = Chem.MolToSmiles(mol, kekuleSmiles=False, isomericSmiles=ISOMERIC)
    hg = tokenizer.tokenize(mol, idx, smiles)
    hg.check_coverage()
    hyper_graphs.append(hg)

# save hyper_graphs
# pickle.dump(hyper_graphs, open(f'{prefix}/train_hyper_graphs.pkl', 'wb'))
